# الفصل الثالث والعشرون: مسار بناء أدوات سطر الأوامر (CLI)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AhmedAlmaghz/Python-Book/blob/main/dist/notebooks/chapter-23.ipynb)

> **الوقت المتوقع:** ~8 دقيقة قراءة · **11** قسمًا · **15** مثالًا برمجيًا.

**كيف تستخدم هذا الدفتر؟**

1. شغّل الخلايا بالترتيب من الأعلى إلى الأسفل (`Shift` + `Enter`).
2. خلايا الشيفرة قابلة للتعديل: غيّر القيم وأعد التشغيل لتجرّب.
3. خلايا 🧾 معروضة للقراءة فقط (أوامر طرفية، أو إدخال يدوي، أو مقتطف ناقص)، وخلايا ⚠️ تُظهر خطأً مقصودًا يُلتقط تلقائيًا فلا يتوقف «تشغيل الكل».

> **Kaggle:** نزّل الملف من المستودع ثم `File ▸ Import Notebook`، أو استورده مباشرة من GitHub بعد ربط حسابك.

<details>
<summary><b>محتويات الفصل</b></summary>

- **23.1** لماذا أدوات سطر الأوامر؟
- **23.2** البداية البدائية: وحدة `argparse` المدمجة
- **23.3** مكتبة `click`: الأساسيات
- **23.4** بناء أوامر فرعية متعددة: `click.group()`
- **23.5** تحويل أداتك إلى أمر حقيقي قابل للتثبيت
- **23.6** أخطاء شائعة وأفضل الممارسات
- **23.7** مشروع تطبيقي متكامل: أداة تنظيم الملفات (File Organizer)
- **23.8** تمارين
- **23.9** اختبر فهمك (Quiz)
- **23.10** ملخص الفصل (Cheat Sheet)
- **23.11** ماذا بعد؟

</details>

In [ ]:
#@title ⚙️ تهيئة البيئة  { display-mode: "form" }
import sys, os, random, pathlib

print("Python:", sys.version.split()[0], "|", sys.platform)
assert sys.version_info >= (3, 10), "هذا الكتاب يفترض Python 3.10 أو أحدث"

SEED = 42
random.seed(SEED)

# تثبيت الحزم الناقصة فقط (سريع إن كانت مثبّتة أصلًا)
import importlib.util as _iu
_need = [m for m in ["click"] if _iu.find_spec(m) is None]
if _need:
    %pip install -q click

# إعداد خاص بهذا الفصل
# الدفتر ليس طرفية: نعطّل كتل __main__ حتى لا تخرج الأدوات بـ SystemExit.
__name__ = 'notebook'

> ℹ️ نكتب الأدوات في ملفات ثم نشغّلها بـ `!python`، تمامًا كما في الطرفية.

## 23.1 لماذا أدوات سطر الأوامر؟

فكّر في كل الأدوات التي استخدمتها حتى الآن في رحلتك: `uv`, `git`, `python`, `uvicorn` — كل هذه أدوات **سطر أوامر (Command-Line Interface - CLI)**. تكتب أمرًا نصيًا، وتحصل على نتيجة فورية، دون الحاجة لفتح نافذة رسومية أو النقر بالفأرة.

هذا ليس أسلوبًا «قديمًا» كما قد يظن البعض — بل هو أسلوب العمل **المفضَّل للمحترفين** في كل المجالات: المبرمجون، مهندسو DevOps، محلّلو البيانات، وحتى المصممون. لماذا؟

> **🔄 مقارنة: أدوات CLI مقابل التطبيقات ذات الواجهة الرسومية (GUI)**
>
> | المعيار | CLI | GUI |
> |---|---|---|
> | السرعة | أسرع بكثير للمستخدم المتمرّس | أبطأ (نقر، تنقّل بين نوافذ) |
> | الأتمتة | سهلة جدًا (يمكن ربطها بسكربتات أخرى) | شبه مستحيلة |
> | الموارد | خفيفة جدًا | تستهلك موارد أكبر |
> | منحنى التعلّم | أصعب في البداية | أسهل للمبتدئين |
> | العمل عن بعد (خوادم) | يعمل دائمًا (SSH) | غالبًا غير متاح |
> | القابلية للتكرار | مثالية (أوامر، سكربتات) | تحتاج تكرار يدوي |

في هذا الفصل، سنتعلّم بناء أدوات CLI احترافية خاصة بنا باستخدام مكتبة **`click`** — إحدى أشهر وأقوى المكتبات لهذا الغرض في عالم بايثون — ثم نبني **أداة تنظيم ملفات متكاملة** تُستخدم فعليًا لترتيب مجلد التنزيلات الفوضوي على أي جهاز.

## 23.2 البداية البدائية: وحدة `argparse` المدمجة

قبل `click`، تأتي مكتبة **`argparse`**، وهي مدمجة في بايثون نفسه (لا تحتاج تثبيت). من المهم معرفتها لأنك ستقابلها كثيرًا في كود الآخرين.

```python
# greet.py
import argparse

parser = argparse.ArgumentParser(description="أداة ترحيب بسيطة")
parser.add_argument("name", help="اسم الشخص المراد الترحيب به")
parser.add_argument("--greeting", default="مرحبًا", help="كلمة الترحيب")
parser.add_argument("--loud", action="store_true", help="اطبع بأحرف كبيرة")

args = parser.parse_args()

message = f"{args.greeting}، {args.name}!"
if args.loud:
    message = message.upper()

print(message)
```

🧾 **للقراءة فقط** — سكربت سطر أوامر يقرأ `sys.argv`؛ شغّله من الطرفية كما في الأمثلة.

```bash
$ python greet.py أحمد
مرحبًا، أحمد!

$ python greet.py أحمد --greeting "أهلاً"
أهلاً، أحمد!

$ python greet.py أحمد --loud
مرحبًا، أحمد!
```

🧾 **للقراءة فقط** — أوامر `bash` تُنفَّذ من الطرفية، لا من الخلية.

`argparse` يعمل ويُنجز المهمة، لكن لاحظ أنه يتطلب كتابة متكررة (boilerplate) كثيرة، ويصعب التوسّع فيه عندما تريد أوامر فرعية متعددة (كما في `git commit` و`git push` — كلاهما أمر فرعي لـ `git`).

> **🔄 مقارنة: `argparse` مقابل `click`**
>
> | المعيار | `argparse` | `click` |
> |---|---|---|
> | التثبيت | مدمج في بايثون | يحتاج `uv add click` |
> | الصياغة | إمبراطورية (imperative)، أكثر تكرارًا | تصريحية (declarative) عبر decorators |
> | الأوامر الفرعية | ممكنة لكن معقّدة الإعداد | بسيطة جدًا عبر `@click.group()` |
> | التحقق من المدخلات | يدوي غالبًا | مدمج (أنواع، نطاقات، خيارات محدودة) |
> | رسائل `--help` | تلقائية لكن بسيطة | تلقائية وأجمل تنسيقًا |
> | الشعبية في المشاريع الحديثة | أقل | أعلى (يُستخدم من Flask, Black, pip...) |
>
> **خلاصة:** `argparse` جيد للسكربتات الصغيرة جدًا والبسيطة. `click` هو الخيار الاحترافي لأي أداة CLI حقيقية ستنمو بمرور الوقت.

## 23.3 مكتبة `click`: الأساسيات

### التثبيت

```bash
uv add click
```

🧾 **للقراءة فقط** — أوامر `bash` تُنفَّذ من الطرفية، لا من الخلية.

### أول أمر بـ click

In [ ]:
# hello.py
import click

@click.command()
def hello():
    """أداة ترحيب بسيطة."""
    click.echo("مرحبًا بك في عالم CLI!")

if __name__ == "__main__":
    hello()

```bash
$ uv run python hello.py
مرحبًا بك في عالم CLI!

$ uv run python hello.py --help
Usage: hello.py [OPTIONS]

  أداة ترحيب بسيطة.

Options:
  --help  Show this message and exit.
```

🧾 **للقراءة فقط** — أوامر `bash` تُنفَّذ من الطرفية، لا من الخلية.

لاحظ كيف أن `@click.command()` حوّلت دالة عادية إلى أمر CLI كامل، بما في ذلك رسالة `--help` **تلقائيًا من الـ docstring**!

> **💡 نصيحة خبير:** استخدم `click.echo()` بدلاً من `print()` العادية داخل أدوات click. السبب: `click.echo()` يتعامل بشكل أفضل وأكثر أمانًا مع الترميزات المختلفة (encoding) عبر أنظمة التشغيل المختلفة (ويندوز، ماك، لينكس)، ويدعم بعض الميزات الإضافية كالألوان.

### المعاملات (Arguments): مدخلات إلزامية بدون اسم

In [ ]:
import click

@click.command()
@click.argument("name")
def greet(name: str):
    """الترحيب باسم محدد."""
    click.echo(f"أهلاً، {name}!")

if __name__ == "__main__":
    greet()

```bash
$ uv run python greet.py أحمد
أهلاً، أحمد!

$ uv run python greet.py
Usage: greet.py [OPTIONS] NAME
Try 'greet.py --help' for help.

Error: Missing argument 'NAME'.
```

🧾 **للقراءة فقط** — أوامر `bash` تُنفَّذ من الطرفية، لا من الخلية.

### الخيارات (Options): مدخلات اختيارية بأسماء `--option`

In [ ]:
import click

@click.command()
@click.argument("name")
@click.option("--greeting", default="مرحبًا", help="كلمة الترحيب المستخدمة")
@click.option("--loud", is_flag=True, help="اطبع الرسالة بأحرف كبيرة")
@click.option("--times", default=1, type=int, help="عدد مرات تكرار الرسالة")
def greet(name: str, greeting: str, loud: bool, times: int):
    """أداة ترحيب قابلة للتخصيص."""
    message = f"{greeting}، {name}!"
    if loud:
        message = message.upper()

    for _ in range(times):
        click.echo(message)

if __name__ == "__main__":
    greet()

```bash
$ uv run python greet.py سارة --greeting "أهلاً وسهلاً" --times 2
أهلاً وسهلاً، سارة!
أهلاً وسهلاً، سارة!

$ uv run python greet.py خالد --loud
مرحبًا، خالد! -> MRHBA، KHALID!  # (مع --loud يصبح بأحرف كبيرة فعليًا للنص اللاتيني)
```

🧾 **للقراءة فقط** — أوامر `bash` تُنفَّذ من الطرفية، لا من الخلية.

> **جدول مرجعي: أنواع الخيارات الشائعة في click**
>
> | الصياغة | الوصف |
> |---|---|
> | `@click.argument("name")` | مدخل إلزامي بدون علامة `--`، بالترتيب |
> | `@click.option("--name")` | مدخل اختياري بعلامة `--`، يقبل قيمة |
> | `is_flag=True` | خيار منطقي (True/False) بدون قيمة، كمفتاح تشغيل/إيقاف |
> | `type=int` / `type=float` | يفرض نوع البيانات ويرفع خطأ واضح إن لم يتطابق |
> | `default=القيمة` | قيمة افتراضية إذا لم يُحدَّد الخيار |
> | `required=True` | يجعل الخيار (وليس Argument) إلزاميًا |
> | `multiple=True` | يسمح بتكرار الخيار عدة مرات، فتُجمَع في tuple |
> | `prompt=True` | يطلب القيمة تفاعليًا من المستخدم إن لم تُحدَّد |

### التحقق المدمج من النوع والنطاق

In [ ]:
import click

@click.command()
@click.option("--age", type=click.IntRange(0, 120), prompt=True, help="العمر")
@click.option(
    "--level",
    type=click.Choice(["مبتدئ", "متوسط", "متقدم"], case_sensitive=False),
    default="مبتدئ",
)
def register(age: int, level: str):
    """تسجيل مستخدم جديد مع تحقق مدمج من المدخلات."""
    click.echo(f"العمر: {age}, المستوى: {level}")

if __name__ == "__main__":
    register()

```bash
$ uv run python register.py --age 200
Usage: register.py [OPTIONS]
Try 'register.py --help' for help.

Error: Invalid value for '--age': 200 is not in the range 0<=x<=120.
```

🧾 **للقراءة فقط** — أوامر `bash` تُنفَّذ من الطرفية، لا من الخلية.

لاحظ أننا حصلنا على **تحقق كامل من صحة البيانات ورسائل خطأ واضحة** دون كتابة أي كود تحقق يدوي — تمامًا كما فعلت Pydantic في الفصل 19!

### قراءة كلمة مرور بأمان

In [ ]:
import click

@click.command()
@click.option("--password", prompt=True, hide_input=True, confirmation_prompt=True)
def set_password(password: str):
    """ضبط كلمة مرور بأمان (مخفية عند الكتابة)."""
    click.echo("✅ تم ضبط كلمة المرور بنجاح!")

if __name__ == "__main__":
    set_password()

### الألوان والتنسيق

```python
import click

click.secho("نجاح! ✅", fg="green", bold=True)
click.secho("تحذير! ⚠️", fg="yellow")
click.secho("خطأ! ❌", fg="red", bold=True)
```

⚠️ **للقراءة فقط** — خطأ مقصود للتوضيح، لذا لا يُنفَّذ.

> **💡 نصيحة خبير:** استخدام الألوان بحكمة في أدوات CLI يُحسّن تجربة المستخدم بشكل كبير — أخضر للنجاح، أصفر للتحذير، أحمر للخطأ، هو اصطلاح عالمي يفهمه كل مستخدم بديهيًا.

## 23.4 بناء أوامر فرعية متعددة: `click.group()`

هذا ما يجعل click قويًا جدًا: بناء أدوات بأوامر فرعية متعددة، تمامًا كـ `git commit`، `git push`، `git status`.

In [ ]:
# cli.py
import click

@click.group()
def cli():
    """أداة إدارة المهام من سطر الأوامر."""
    pass

@cli.command()
@click.argument("task")
def add(task: str):
    """إضافة مهمة جديدة."""
    click.echo(f"✅ تمت إضافة المهمة: {task}")

@cli.command()
def list():
    """عرض كل المهام."""
    click.echo("📋 قائمة المهام:")
    click.echo("1. تعلّم click")
    click.echo("2. بناء مشروع CLI")

@cli.command()
@click.argument("task_id", type=int)
def done(task_id: int):
    """تحديد مهمة كمنجزة."""
    click.echo(f"🎉 تم إنجاز المهمة رقم {task_id}")

if __name__ == "__main__":
    cli()

```bash
$ uv run python cli.py --help
Usage: cli.py [OPTIONS] COMMAND [ARGS]...

  أداة إدارة المهام من سطر الأوامر.

Options:
  --help  Show this message and exit.

Commands:
  add   إضافة مهمة جديدة.
  done  تحديد مهمة كمنجزة.
  list  عرض كل المهام.

$ uv run python cli.py add "شراء القهوة"
✅ تمت إضافة المهمة: شراء القهوة

$ uv run python cli.py done 1
🎉 تم إنجاز المهمة رقم 1
```

🧾 **للقراءة فقط** — أوامر `bash` تُنفَّذ من الطرفية، لا من الخلية.

لاحظ النمط: `@click.group()` يُنشئ «الأداة الأم»، ثم `@cli.command()` (وليس `@click.command()`) يُضيف كل أمر فرعي إليها.

## 23.5 تحويل أداتك إلى أمر حقيقي قابل للتثبيت

حتى الآن كنا نُشغّل الأداة بـ `uv run python cli.py`. لكن أدوات CLI الحقيقية (مثل `git`) تُستدعى مباشرة بـ `git` فقط. يمكننا تحقيق هذا عبر `pyproject.toml`:

```toml
[project.scripts]
mytasks = "my_package.cli:cli"
```

🧾 **للقراءة فقط** — محتوى ملف `toml`.

هذا يُخبر أداة التثبيت: "أنشئ أمرًا باسم `mytasks`، يستدعي الدالة `cli` من الوحدة `my_package.cli`". بعد `uv sync` أو التثبيت، يمكنك كتابة:

```bash
$ mytasks add "شراء القهوة"
$ mytasks list
```

🧾 **للقراءة فقط** — أوامر `bash` تُنفَّذ من الطرفية، لا من الخلية.

مباشرة من أي مكان في الطرفية، تمامًا كأي أمر نظام حقيقي!

> **💡 نصيحة خبير:** هذا بالضبط كيف تُبنى أدوات مثل `black`, `ruff`, `uv` نفسها! كلها حزم بايثون عادية، تُعرِّف `[project.scripts]` في `pyproject.toml` لتصبح أوامر نظام قابلة للاستدعاء المباشر.

## 23.6 أخطاء شائعة وأفضل الممارسات

> **⚠️ تحذير: نسيان `type hints` ومعالجة حالات الحافة**
> أدوات CLI تتفاعل مع **مستخدمين حقيقيين** يُدخلون بيانات غير متوقعة. تعامل دائمًا مع:
> - مسارات ملفات غير موجودة
> - صلاحيات وصول مرفوضة
> - مدخلات فارغة أو بتنسيق خاطئ
>
> ```python
> import click
> from pathlib import Path
>
> @click.command()
> @click.argument("file_path", type=click.Path(exists=True, path_type=Path))
> def process(file_path: Path):
>     ""«معالجة ملف -- click يتحقق من وجوده تلقائيًا!»""
>     click.echo(f«معالجة: {file_path}»)
> ```
> لاحظ `type=click.Path(exists=True)` — يتحقق click تلقائيًا من وجود الملف، ويرفض الاستمرار برسالة خطأ واضحة إن لم يكن موجودًا، بدلاً من أن ينهار برنامجك لاحقًا بخطأ غامض.

> **⚠️ تحذير: عمليات خطيرة بدون تأكيد**
> أي أمر قد يحذف أو يُعدّل بيانات بشكل لا رجعة فيه يجب أن يطلب تأكيدًا:
> ```python
> @click.command()
> @click.confirmation_option(prompt=«هل أنت متأكد من حذف كل الملفات؟»)
> def delete_all():
>     click.echo(«تم الحذف.»)
> ```
> ```bash
> $ python cli.py delete_all
> هل أنت متأكد من حذف كل الملفات؟ [y/N]: n
> Aborted!
> ```

> **💡 نصيحة خبير: استخدام `click.progressbar` للعمليات الطويلة**
> ```python
> import time
> import click
>
> items = range(100)
> with click.progressbar(items, label=«معالجة الملفات») as bar:
>     for item in bar:
>         time.sleep(0.01)  # محاكاة عمل فعلي
> ```
> يُعطي المستخدم تغذية راجعة بصرية بدلاً من ترك البرنامج «صامتًا» لعدة ثوانٍ أو دقائق، مما يُحسّن تجربة الاستخدام بشكل كبير.

## 23.7 مشروع تطبيقي متكامل: أداة تنظيم الملفات (File Organizer)

سنبني أداة CLI احترافية حقيقية: تفحص مجلدًا (مثل مجلد التنزيلات)، وتُنظّم ملفاته تلقائيًا في مجلدات فرعية حسب نوعها (صور، مستندات، فيديوهات...)، مع خيارات متعددة ووضع «تجربة» آمن قبل التنفيذ الفعلي.

### هيكلية المشروع

```
file_organizer/
├── pyproject.toml
└── src/
    └── file_organizer/
        ├── __init__.py
        ├── rules.py
        ├── organizer.py
        └── cli.py
```

```bash
uv init file_organizer
cd file_organizer
uv add click
mkdir -p src/file_organizer
touch src/file_organizer/__init__.py
```

🧾 **للقراءة فقط** — أوامر `bash` تُنفَّذ من الطرفية، لا من الخلية.

### `src/file_organizer/rules.py` — قواعد تصنيف الملفات

```python
"""قواعد تصنيف أنواع الملفات إلى فئات."""

FILE_CATEGORIES: dict[str, list[str]] = {
    "الصور": [".jpg", ".jpeg", ".png", ".gif", ".svg", ".webp", ".bmp"],
    "المستندات": [".pdf", ".docx", ".doc", ".txt", ".md", ".xlsx", ".pptx"],
    "الفيديوهات": [".mp4", ".mkv", ".avi", ".mov", ".webm"],
    "الصوتيات": [".mp3", ".wav", ".flac", ".aac"],
    "الأرشيف": [".zip", ".rar", ".tar", ".gz", ".7z"],
    "الأكواد": [".py", ".js", ".html", ".css", ".json", ".java", ".cpp"],
}

DEFAULT_CATEGORY = "أخرى"

def get_category_for_file(extension: str) -> str:
    """تحديد الفئة المناسبة لامتداد ملف معيّن.

    Args:
        extension: امتداد الملف (مثل '.jpg'), بأي حالة أحرف.

    Returns:
        اسم الفئة المناسبة، أو DEFAULT_CATEGORY إن لم يُعرف الامتداد.
    """
    extension = extension.lower()
    for category, extensions in FILE_CATEGORIES.items():
        if extension in extensions:
            return category
    return DEFAULT_CATEGORY
```

🧾 **للقراءة فقط** — محتوى ملف داخل المشروع؛ يُكتب على القرص ثم يُشغَّل من الطرفية.

### `src/file_organizer/organizer.py` — منطق التنظيم الأساسي

```python
"""منطق فحص وتنظيم الملفات في مجلد."""

import shutil
from dataclasses import dataclass
from pathlib import Path

from file_organizer.rules import get_category_for_file

@dataclass
class FileMove:
    """يمثّل عملية نقل ملف واحد مخطَّط لها."""

    source: Path
    destination: Path
    category: str

def scan_directory(directory: Path) -> list[FileMove]:
    """فحص مجلد وتحديد وجهة كل ملف فيه دون تنفيذ أي نقل فعلي.

    Args:
        directory: المجلد المراد فحصه.

    Returns:
        قائمة بعمليات النقل المخطَّط لها (FileMove) لكل ملف.
    """
    planned_moves: list[FileMove] = []

    for item in directory.iterdir():
        if item.is_dir():
            continue  # نتجاهل المجلدات الفرعية الموجودة مسبقًا

        category = get_category_for_file(item.suffix)
        destination_folder = directory / category
        destination_path = destination_folder / item.name

        planned_moves.append(
            FileMove(source=item, destination=destination_path, category=category)
        )

    return planned_moves

def execute_moves(moves: list[FileMove]) -> tuple[int, list[str]]:
    """تنفيذ عمليات النقل المخطَّط لها فعليًا.

    Args:
        moves: قائمة عمليات النقل المطلوب تنفيذها.

    Returns:
        عدد العمليات الناجحة، وقائمة برسائل الأخطاء إن وُجدت.
    """
    success_count = 0
    errors: list[str] = []

    for move in moves:
        try:
            move.destination.parent.mkdir(exist_ok=True)

            if move.destination.exists():
                # تجنّب الكتابة فوق ملف موجود بنفس الاسم
                errors.append(f"⚠️ تخطّي {move.source.name}: يوجد ملف بنفس الاسم في الوجهة")
                continue

            shutil.move(str(move.source), str(move.destination))
            success_count += 1

        except OSError as e:
            errors.append(f"❌ فشل نقل {move.source.name}: {e}")

    return success_count, errors

def summarize_by_category(moves: list[FileMove]) -> dict[str, int]:
    """تلخيص عدد الملفات لكل فئة."""
    summary: dict[str, int] = {}
    for move in moves:
        summary[move.category] = summary.get(move.category, 0) + 1
    return summary
```

🧾 **للقراءة فقط** — محتوى ملف داخل المشروع؛ يُكتب على القرص ثم يُشغَّل من الطرفية.

> **💡 نصيحة خبير:** لاحظ الفصل الواضح بين `scan_directory` (التخطيط فقط، بدون أي تعديل فعلي على نظام الملفات) و`execute_moves` (التنفيذ الفعلي). هذا الفصل يُمكّننا بسهولة من بناء ميزة **«وضع التجربة» (Dry Run)** — عرض ما سيحدث **قبل** أن يحدث فعليًا، وهي ميزة أساسية في أي أداة تتعامل مع عمليات لا رجعة فيها.

### `src/file_organizer/cli.py` — واجهة سطر الأوامر

```python
"""واجهة سطر الأوامر لأداة تنظيم الملفات."""

from pathlib import Path

import click

from file_organizer.organizer import execute_moves, scan_directory, summarize_by_category

@click.group()
@click.version_option(version="1.0.0")
def cli():
    """🗂️ أداة تنظيم الملفات تلقائيًا حسب نوعها."""
    pass

@cli.command()
@click.argument(
    "directory",
    type=click.Path(exists=True, file_okay=False, path_type=Path),
)
def preview(directory: Path):
    """معاينة خطة التنظيم دون تنفيذ أي تغيير فعلي (وضع آمن)."""
    moves = scan_directory(directory)

    if not moves:
        click.secho("📭 لا توجد ملفات لتنظيمها في هذا المجلد.", fg="yellow")
        return

    click.secho(f"📋 خطة تنظيم {len(moves)} ملف:\n", bold=True)

    summary = summarize_by_category(moves)
    for category, count in summary.items():
        click.echo(f"  📁 {category}: {count} ملف")

    click.echo()
    click.secho(
        "💡 هذه معاينة فقط. استخدم الأمر 'organize' لتنفيذ التغييرات فعليًا.",
        fg="cyan",
    )

@cli.command()
@click.argument(
    "directory",
    type=click.Path(exists=True, file_okay=False, path_type=Path),
)
@click.confirmation_option(
    prompt="⚠️ سيتم نقل الملفات فعليًا. هل أنت متأكد؟"
)
def organize(directory: Path):
    """تنظيم الملفات فعليًا في المجلد المحدد حسب نوعها."""
    moves = scan_directory(directory)

    if not moves:
        click.secho("📭 لا توجد ملفات لتنظيمها في هذا المجلد.", fg="yellow")
        return

    click.echo(f"🚀 جاري تنظيم {len(moves)} ملف...\n")

    with click.progressbar(moves, label="التقدّم") as bar:
        success_count, errors = execute_moves(list(bar))

    click.echo()
    click.secho(f"✅ تم نقل {success_count} ملف بنجاح!", fg="green", bold=True)

    if errors:
        click.secho(f"\n⚠️ حدثت {len(errors)} مشكلة:", fg="yellow")
        for error in errors:
            click.echo(f"  {error}")

@cli.command()
@click.argument(
    "directory",
    type=click.Path(exists=True, file_okay=False, path_type=Path),
)
def stats(directory: Path):
    """عرض إحصائيات عن أنواع الملفات الموجودة في المجلد."""
    moves = scan_directory(directory)
    summary = summarize_by_category(moves)

    if not summary:
        click.secho("📭 المجلد فارغ.", fg="yellow")
        return

    total = sum(summary.values())
    click.secho(f"📊 إحصائيات المجلد ({total} ملف إجمالاً):\n", bold=True)

    for category, count in sorted(summary.items(), key=lambda x: x[1], reverse=True):
        percentage = (count / total) * 100
        bar = "█" * int(percentage / 5)
        click.echo(f"  {category:12s} {bar} {count} ({percentage:.1f}%)")

if __name__ == "__main__":
    cli()
```

🧾 **للقراءة فقط** — محتوى ملف داخل المشروع؛ يُكتب على القرص ثم يُشغَّل من الطرفية.

### ربط الأداة كأمر نظام حقيقي: `pyproject.toml`

```toml
[project]
name = "file-organizer"
version = "1.0.0"
requires-python = ">=3.14"
dependencies = [
    "click>=8.1",
]

[project.scripts]
forganize = "file_organizer.cli:cli"

[build-system]
requires = ["hatchling"]
build-backend = "hatchling.build"
```

🧾 **للقراءة فقط** — محتوى ملف `toml`.

تثبيت الأداة محليًا للتجربة:

```bash
uv sync
uv run forganize --help
```

🧾 **للقراءة فقط** — أوامر `bash` تُنفَّذ من الطرفية، لا من الخلية.

```
Usage: forganize [OPTIONS] COMMAND [ARGS]...

  🗂️ أداة تنظيم الملفات تلقائيًا حسب نوعها.

Options:
  --version  Show the version and exit.
  --help     Show this message and exit.

Commands:
  organize  تنظيم الملفات فعليًا في المجلد المحدد حسب نوعها.
  preview   معاينة خطة التنظيم دون تنفيذ أي تغيير فعلي (وضع آمن).
  stats     عرض إحصائيات عن أنواع الملفات الموجودة في المجلد.
```

### تجربة الاستخدام الكاملة

```bash
$ uv run forganize stats ~/Downloads
📊 إحصائيات المجلد (47 ملف إجمالاً):

  الصور        ████████████ 23 (48.9%)
  المستندات    ██████ 12 (25.5%)
  الأرشيف      ███ 7 (14.9%)
  أخرى         ██ 5 (10.6%)

$ uv run forganize preview ~/Downloads
📋 خطة تنظيم 47 ملف:

  📁 الصور: 23 ملف
  📁 المستندات: 12 ملف
  📁 الأرشيف: 7 ملف
  📁 أخرى: 5 ملف

💡 هذه معاينة فقط. استخدم الأمر 'organize' لتنفيذ التغييرات فعليًا.

$ uv run forganize organize ~/Downloads
⚠️ سيتم نقل الملفات فعليًا. هل أنت متأكد؟ [y/N]: y
🚀 جاري تنظيم 47 ملف...

التقدّم  [####################################]  100%

✅ تم نقل 47 ملف بنجاح!
```

🧾 **للقراءة فقط** — أوامر `bash` تُنفَّذ من الطرفية، لا من الخلية.

<details>
<summary>✅ الحل الكامل للمشروع (جميع الملفات مجتمعة)</summary>

جميع ملفات `rules.py`، `organizer.py`، `cli.py` كما وردت بالكامل أعلاه، مع `pyproject.toml` المُحدَّث بـ `[project.scripts]`.

خطوات الإعداد الكاملة:

```bash
uv init file_organizer
cd file_organizer
uv add click
# أنشئ الملفات كما هو موضّح، ثم:
uv sync
uv run forganize --help
```

🧾 **للقراءة فقط** — أوامر `bash` تُنفَّذ من الطرفية، لا من الخلية.

**نقاط تصميم مهمة في هذا الحل:**
1. **فصل التخطيط عن التنفيذ**: `scan_directory` (آمن، بدون آثار جانبية) منفصل تمامًا عن `execute_moves` (ينفّذ فعليًا).
2. **وضع المعاينة (`preview`)**: يسمح للمستخدم برؤية ما سيحدث قبل التنفيذ — ممارسة أساسية في أي أداة تُجري تغييرات لا رجعة فيها.
3. **تأكيد إلزامي (`@click.confirmation_option`)** قبل تنفيذ عملية لا يمكن التراجع عنها بسهولة.
4. **معالجة أخطاء دفاعية**: تخطّي الملفات التي قد تسبب تعارضًا (اسم مكرر) بدلاً من إيقاف العملية بالكامل أو حذف بيانات بالخطأ.
5. **تحويلها لأمر نظام حقيقي** عبر `[project.scripts]` — نفس الآلية المستخدمة في كل أدوات بايثون الاحترافية.

</details>

## 23.8 تمارين

### 🟢 مستوى مبتدئ

1. اكتب أمر click بسيط `@click.command()` اسمه `square` يأخذ رقمًا `@click.argument("number", type=int)` ويطبع مربعه.
2. أضف خيارًا `--unit` لأمر يحوّل درجات الحرارة، بقيمة افتراضية `"celsius"`، واستخدم `click.Choice(["celsius", "fahrenheit"])`.
3. استخدم `click.confirmation_option` على أمر وهمي اسمه `reset` يطبع «تم إعادة الضبط!» بعد التأكيد.

<details>
<summary>✅ الحل المقترح</summary>

In [ ]:
import click

# 1
@click.command()
@click.argument("number", type=int)
def square(number: int):
    click.echo(f"{number}² = {number ** 2}")

# 2
@click.command()
@click.argument("value", type=float)
@click.option("--unit", type=click.Choice(["celsius", "fahrenheit"]), default="celsius")
def convert_temp(value: float, unit: str):
    if unit == "celsius":
        result = value * 9 / 5 + 32
        click.echo(f"{value}°C = {result}°F")
    else:
        result = (value - 32) * 5 / 9
        click.echo(f"{value}°F = {result}°C")

# 3
@click.command()
@click.confirmation_option(prompt="هل تريد إعادة الضبط؟")
def reset():
    click.echo("تم إعادة الضبط!")

if __name__ == "__main__":
    square()

</details>

### 🟡 مستوى متوسط

4. أضف أمرًا فرعيًا جديدًا إلى أداة تنظيم الملفات اسمه `undo` يبحث عن الملفات الموجودة في مجلدات الفئات (الصور، المستندات...) ويعيدها إلى المجلد الأصلي (عكس عملية `organize`).
5. أضف خيار `--dry-run` (كـ flag) إلى أمر `organize` نفسه، بدلاً من أمر `preview` منفصل، بحيث يقوم بنفس وظيفة المعاينة عند تفعيله.

<details>
<summary>✅ الحل المقترح</summary>

```python
# تمرين 5 -- دمج المعاينة كخيار بدلاً من أمر منفصل
@cli.command()
@click.argument("directory", type=click.Path(exists=True, file_okay=False, path_type=Path))
@click.option("--dry-run", is_flag=True, help="معاينة فقط دون تنفيذ فعلي")
def organize(directory: Path, dry_run: bool):
    """تنظيم الملفات، مع إمكانية المعاينة عبر --dry-run."""
    moves = scan_directory(directory)

    if not moves:
        click.secho("📭 لا توجد ملفات لتنظيمها.", fg="yellow")
        return

    if dry_run:
        click.secho("🔍 وضع المعاينة (dry-run) -- لن يتم تنفيذ أي تغيير:\n", fg="cyan")
        summary = summarize_by_category(moves)
        for category, count in summary.items():
            click.echo(f"  📁 {category}: {count} ملف")
        return

    # التنفيذ الفعلي كما في الكود الأصلي...
    success_count, errors = execute_moves(moves)
    click.secho(f"✅ تم نقل {success_count} ملف!", fg="green")
```

🧾 **للقراءة فقط** — مثال مقتطف يعتمد على `Path`، `execute_moves`، `scan_directory` من مثال سابق.

(تمرين 4 يتطلب دالة عكسية تمسح المجلدات الفرعية المعروفة في `FILE_CATEGORIES.keys()` وتُعيد كل ملف لمستوى المجلد الأصلي باستخدام نفس نمط `shutil.move`.)

</details>

### 🔴 مستوى تحدي

6. وسّع الأداة لدعم **ملف إعدادات مخصص** (`.forganizerc` بصيغة JSON أو TOML) يسمح للمستخدم بتعريف فئات ملفات خاصة به (مثلاً فئة «عمل» لملفات `.pptx` و`.key`)، بدلاً من الاعتماد فقط على `FILE_CATEGORIES` الثابتة في `rules.py`. الأداة يجب أن تبحث عن هذا الملف في المجلد الحالي أو مجلد المستخدم الرئيسي (`Path.home()`) وتدمج إعداداته مع الإعدادات الافتراضية.

<details>
<summary>✅ الحل المقترح (تصور)</summary>

```python
import json
from pathlib import Path

from file_organizer.rules import FILE_CATEGORIES

def load_custom_rules() -> dict[str, list[str]]:
    """تحميل قواعد تصنيف مخصصة من ملف إعدادات إن وُجد، ودمجها مع الافتراضية."""
    config_locations = [Path.cwd() / ".forganizerc", Path.home() / ".forganizerc"]

    merged_rules = dict(FILE_CATEGORIES)  # نسخة من القواعد الافتراضية

    for config_path in config_locations:
        if config_path.exists():
            custom_rules = json.loads(config_path.read_text(encoding="utf-8"))
            merged_rules.update(custom_rules)  # القواعد المخصصة تطغى على الافتراضية
            break

    return merged_rules
```

🧾 **للقراءة فقط** — الوحدة `file_organizer` تُنشأ داخل مشروع محلي أو ليست حزمة عامة، فلا يمكن استيرادها هنا.

مثال ملف `.forganizerc`:

```json
{
  "عمل": [".pptx", ".key", ".pages"]
}
```

🧾 **للقراءة فقط** — محتوى ملف `json`.

</details>

## 23.9 اختبر فهمك (Quiz)

<details>
<summary>1. ما الفرق بين `@click.argument` و `@click.option`؟</summary>

`@click.argument` يُعرّف مدخلاً **إلزاميًا** يُمرَّر بالترتيب بدون اسم `--` (مثل `greet أحمد`)، بينما `@click.option` يُعرّف مدخلاً **اختياريًا** (عادة) يُمرَّر باسم صريح يبدأ بـ `--` (مثل `--greeting مرحبًا`)، ويمكن أن يحمل قيمة افتراضية.
</details>

<details>
<summary>2. كيف تبني أداة بأوامر فرعية متعددة مثل git؟</summary>

باستخدام `@click.group()` لتعريف «الأداة الأم»، ثم تعريف كل أمر فرعي بـ `@اسم_المجموعة.command()` بدلاً من `@click.command()` مباشرة، بحيث تصبح الأوامر الفرعية قابلة للاستدعاء كـ `الأداة الأم الأمر_الفرعي`.
</details>

<details>
<summary>3. لماذا نستخدم `click.Path(exists=True)` بدلاً من `str` عادي للمسارات؟</summary>

لأن click يتحقق تلقائيًا من وجود المسار فعليًا على القرص قبل تمريره للدالة، ويرفض الاستمرار برسالة خطأ واضحة ومفهومة للمستخدم إذا كان المسار غير موجود، بدلاً من أن يفشل البرنامج لاحقًا في منتصف التنفيذ بخطأ غامض مثل `FileNotFoundError`.
</details>

<details>
<summary>4. ما الفائدة من فصل «التخطيط» (scan/preview) عن «التنفيذ» (execute) في أداة مثل منظّم الملفات؟</summary>

يسمح هذا الفصل بتوفير وضع معاينة آمن (Dry Run) يُظهر للمستخدم بالضبط ما سيحدث **قبل** حدوثه فعليًا، دون أي خطر على بياناته. هذا مهم جدًا في أي أداة تُجري تغييرات لا يمكن التراجع عنها بسهولة (كنقل أو حذف ملفات)، ويمنح المستخدم الثقة في استخدام الأداة.
</details>

<details>
<summary>5. كيف تجعل أداة CLI التي بنيتها قابلة للاستدعاء مباشرة كأمر نظام (مثل git) بدلاً من `python script.py`؟</summary>

عبر تعريف قسم `[project.scripts]` في ملف `pyproject.toml`، يربط اسم الأمر المطلوب بالدالة التي تمثّل نقطة الدخول (مثل `cli` المُزينة بـ `@click.group()`)، ثم تثبيت الحزمة (عبر `uv sync` أو `pip install`)، فيصبح الأمر متاحًا مباشرة في الطرفية.
</details>

## 23.10 ملخص الفصل (Cheat Sheet)

```python
import click

# ===== أمر بسيط =====
@click.command()
def hello():
    """وصف الأمر (يظهر في --help)."""
    click.echo("مرحبًا!")

# ===== Argument مقابل Option =====
@click.argument("name")                          # إلزامي، بدون --
@click.option("--greeting", default="مرحبًا")     # اختياري، بـ --
@click.option("--loud", is_flag=True)             # علم منطقي True/False
@click.option("--age", type=click.IntRange(0, 120))
@click.option("--level", type=click.Choice(["a", "b"]))

# ===== أوامر فرعية (مثل git) =====
@click.group()
def cli():
    pass

@cli.command()
def sub_command():
    pass

# ===== ميزات مفيدة =====
click.secho("نص ملوّن", fg="green", bold=True)
click.confirmation_option(prompt="متأكد؟")
click.progressbar(items, label="جاري العمل")
type=click.Path(exists=True, path_type=Path)

# ===== تحويلها لأمر نظام =====
# في pyproject.toml:
# [project.scripts]
# mytool = "my_package.cli:cli"
```

🧾 **للقراءة فقط** — مثال مقتطف يعتمد على `Path`، `items` من مثال سابق.

| المهمة | الحل |
|---|---|
| مدخل إلزامي | `@click.argument("x")` |
| مدخل اختياري بقيمة افتراضية | `@click.option("--x", default=...)` |
| علم تشغيل/إيقاف | `@click.option("--x", is_flag=True)` |
| تحقق من نوع/نطاق | `type=int`, `type=click.IntRange(...)` |
| أوامر فرعية | `@click.group()` + `@cli.command()` |
| تأكيد قبل عملية خطيرة | `@click.confirmation_option()` |
| شريط تقدّم | `click.progressbar(...)` |

## 23.11 ماذا بعد؟

لقد بنيت الآن أداة CLI احترافية وحقيقية — من النوع الذي يمكنك نشره فعليًا واستخدامه يوميًا، أو مشاركته مع زملائك. بهذا، تكون قد أتممت **الجزء الرابع: جسور التخصص** بالكامل، وتعرّفت عمليًا على خمسة مسارات مختلفة لبايثون: الويب، تحليل البيانات، الأتمتة، الذكاء الاصطناعي، وأدوات سطر الأوامر.

الآن، ننتقل إلى **الجزء الخامس: الاحتراف والجودة** — حيث سنرتقي بكل ما تعلمته إلى مستوى احترافي حقيقي. نبدأ بالفصل القادم: **كتابة كود نظيف (Clean Code)**، حيث سنتعلّم معايير PEP 8 بعمق، أدوات الفحص الآلي مثل `black` و`ruff`، ومبادئ SOLID المبسّطة لكتابة كود يفهمه ويُحبّه كل من يقرأه بعدك — بما في ذلك أنت نفسك بعد ستة أشهر!

---

### الخطوة التالية

- [الفصل الرابع والعشرون: كتابة كود نظيف (Clean Code)](https://colab.research.google.com/github/AhmedAlmaghz/Python-Book/blob/main/dist/notebooks/chapter-24.ipynb)
- [فهرس كل الدفاتر](https://colab.research.google.com/github/AhmedAlmaghz/Python-Book/blob/main/dist/notebooks/00-index.ipynb)

*الفصل الثالث والعشرون — من كتاب «بايثون 3.14: من الصفر إلى الاحتراف».*